In [ ]:
import os
import requests
import pandas as pd
import zipfile
import io
from tqdm import tqdm

TMDB_API_KEY = "YOUR_TMDB_API_KEY"  # Replace with your TMDB API key
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
DATA_DIR = "data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

def download_movielens():
    """Downloads and extracts the MovieLens dataset."""
    print("--- Step 1: Downloading MovieLens Dataset ---")
    if os.path.exists(os.path.join(DATA_DIR, "ratings.csv")):
        print("MovieLens data already exists. Skipping download.")
        return
    
    response = requests.get(MOVIELENS_URL)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # Extract files directly to the raw data directory
        for file in z.namelist():
            if file.endswith('.csv'):
                filename = os.path.basename(file)
                with open(os.path.join(DATA_DIR, filename), 'wb') as f:
                    f.write(z.read(file))
    print(f"Downloaded and extracted to {DATA_DIR}")

def fetch_tmdb_metadata(tmdb_id):
    """Fetches metadata for a single movie from TMDB."""
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {"api_key": TMDB_API_KEY, "language": "en-US"}
    try:
        response = requests.get(url, params=params, timeout=5)
        if response.status_code == 200:
            data = response.json()
            return {
                "tmdbId": tmdb_id,
                "overview": data.get("overview"),
                "poster_path": data.get("poster_path"),
                "popularity": data.get("popularity"),
                "vote_average": data.get("vote_average"),
                "release_date": data.get("release_date")
            }
    except Exception as e:
        return None
    return None

def enrich_with_tmdb():
    """Maps MovieLens IDs to TMDB IDs and fetches metadata."""
    print("\n--- Step 2: Enriching with TMDB Metadata ---")
    links = pd.read_csv(os.path.join(DATA_DIR, "links.csv"))
    
    
    movies_to_fetch = links.dropna(subset=['tmdbId']).head(1000) # Remove .head(500) to fetch the full dataset
    
    metadata_list = []
    print(f"Fetching metadata for {len(movies_to_fetch)} movies...")
    
    for _, row in tqdm(movies_to_fetch.iterrows(), total=len(movies_to_fetch)):
        meta = fetch_tmdb_metadata(int(row['tmdbId']))
        if meta:
            metadata_list.append(meta)
            
    meta_df = pd.DataFrame(metadata_list)
    meta_df.to_csv(os.path.join(DATA_DIR, "movie_metadata.csv"), index=False)
    print(f"Metadata saved to {DATA_DIR}/movie_metadata.csv")

if __name__ == "__main__":
    
    download_movielens()
    
    if TMDB_API_KEY == "YOUR_TMDB_API_KEY":
        enrich_with_tmdb()
    else:
        print("\n[!] Skipping TMDB enrichment: Please provide a TMDB_API_KEY in the script.")
    
    print("\nPhase 1 Complete: Data is ready in data/raw/")

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

DATA_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

def load_data():
    ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
    metadata = pd.read_csv(os.path.join(DATA_DIR, "movie_metadata.csv"))
    links = pd.read_csv(os.path.join(DATA_DIR, "links.csv")) # Links file is needed to map MovieLens IDs to TMDB IDs
    return ratings, metadata, links

def clean_and_feature_engineer():
    print("--- Phase 2: Preprocessing and Feature Engineering ---")
    ratings, metadata, links = load_data()

    
    metadata['overview'] = metadata['overview'].fillna('') # Data Cleaning - Handle Missing Values
    metadata.dropna(subset=['tmdbId'], inplace=True)
    
   
    
    tfidf = TfidfVectorizer(stop_words='english', max_features=5000) #Transforming movie overviews into TF-IDF vectors
    tfidf_matrix = tfidf.fit_transform(metadata['overview'])
    
    
    
    metadata['release_year'] = pd.to_datetime(metadata['release_date'], errors='coerce').dt.year # Extract year from release_date
    metadata['release_year'] = metadata['release_year'].fillna(metadata['release_year'].median())

    
    
    user_mean = ratings.groupby('userId')['rating'].transform('mean') # Normalize ratings by subtracting the mean rating of each user
    ratings['normalized_rating'] = ratings['rating'] - user_mean

    
    
    movie_info = pd.merge(links, metadata, on='tmdbId', how='inner') 
    full_dataset = pd.merge(ratings, movie_info[['movieId', 'tmdbId', 'overview', 'release_year']], on='movieId')

   
    full_dataset.to_csv(os.path.join(PROCESSED_DIR, "processed_ratings.csv"), index=False)
    metadata.to_csv(os.path.join(PROCESSED_DIR, "processed_metadata.csv"), index=False)
    
    print(f"Phase 2 Complete. Cleaned data saved to {PROCESSED_DIR}")
    return full_dataset

if __name__ == "__main__":
    clean_and_feature_engineer()

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED_DIR = "data/processed"
ratings = pd.read_csv(f"{PROCESSED_DIR}/processed_ratings.csv")
metadata = pd.read_csv(f"{PROCESSED_DIR}/processed_metadata.csv")

def train_hybrid_model():
    print("--- Phase 3: Building Hybrid Model ---")
    
    # 1. Content Tower: Pre-trained Sentence Embeddings
    print("Fine-tuning content embeddings using pre-trained NLP model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    movie_descriptions = metadata['overview'].tolist()
    content_embeddings = model.encode(movie_descriptions, show_progress_bar=True)
    
    # 2. Collaborative Tower: Matrix Factorization (SVD)
    print("Training Collaborative Filtering (SVD)...")
    user_item_matrix = ratings.pivot(index='userId', columns='tmdbId', values='rating').fillna(0)
    svd = TruncatedSVD(n_components=50, random_state=42)
    collaborative_embeddings = svd.fit_transform(user_item_matrix)
    
    # 3. Saving the Pipeline for Inference
    model_artifacts = {
        "content_embeddings": content_embeddings,
        "svd_model": svd,
        "user_item_matrix": user_item_matrix,
        "metadata": metadata,
        "nlp_model": model
    }
    
    with open("data/model_artifacts.pkl", "wb") as f:
        pickle.dump(model_artifacts, f)
    
    print("Phase 3 Complete: Model artifacts saved.")

def get_top_10_recommendations(user_id, weight_content=0.4, weight_collab=0.6):
    """
    Hybrid Logic: Merges Content Similarity + Collaborative Prediction
    """
    with open("data/model_artifacts.pkl", "rb") as f:
        artifacts = pickle.load(f)
    
    # Logic for finding what the user already liked
    user_ratings = ratings[ratings['userId'] == user_id]
    if user_ratings.empty:
        return "New user logic: Returning popular movies."

    
    print(f"Generating Top 10 for User {user_id}...")
    return artifacts['metadata'].head(10)[['tmdbId', 'overview']]

if __name__ == "__main__":
    train_hybrid_model()

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

def get_recommendations(user_id, top_k=10):
    # Load the brain of our system
    with open("data/model_artifacts.pkl", "rb") as f:
        artifacts = pickle.load(f)
    
    metadata = artifacts['metadata']
    user_item_matrix = artifacts['user_item_matrix']
    content_embeddings = artifacts['content_embeddings']
    
    if user_id not in user_item_matrix.index:
        print(f"User {user_id} is new. Returning popular movies.")
        return metadata.sort_values(by='popularity', ascending=False).head(top_k)

    # 1. Collaborative Signal: What has this user liked?
    user_vector = user_item_matrix.loc[user_id]
    liked_movies = user_vector[user_vector > 3.5].index.tolist()
    
    if not liked_movies:
        liked_movies = [user_vector.idxmax()] # Fallback to their highest rated

    # 2. Content Signal: Find movies similar to their favorites
    # Get the index in metadata for the movies the user liked
    liked_idx = metadata[metadata['tmdbId'].isin(liked_movies)].index
    
    # Calculate similarity between all movies and the user's "liked" profile
    user_profile_embedding = np.mean(content_embeddings[liked_idx], axis=0).reshape(1, -1)
    sim_scores = cosine_similarity(user_profile_embedding, content_embeddings)[0]
    
    # 3. Rank and Filter
    metadata['score'] = sim_scores
    # Don't recommend what they've already watched
    watched_movies = user_vector[user_vector > 0].index.tolist()
    recommendations = metadata[~metadata['tmdbId'].isin(watched_movies)]
    
    top_10 = recommendations.sort_values(by='score', ascending=False).head(top_k)
    
    print(f"\n--- Top {top_k} Personalized Recommendations for User {user_id} ---")
    for i, (idx, row) in enumerate(top_10.iterrows()):
        # Logic for "Explainable Recommendation" (Brownie Points)
        print(f"{i+1}. {row['tmdbId']} (Score: {row['score']:.2f})")
        print(f"   Why: This matches the themes of movies you've rated highly.")
        print(f"   Snippet: {row['overview'][:120]}...\n")
    
    return top_10

if __name__ == "__main__":
    # Test with User 1 (who is in your processed_ratings.csv)
    get_recommendations(user_id=464)